# ScriptFlow — writer-identification training (v2)

Trains the embedding model the backend serves (`ml/model.py`) and exports a drop-in
`best_model.pth`. Built from `iam_pretrain_finale_1.ipynb`, keeping what worked there
(height-224 preprocessing, 4-crop averaging, BatchNorm-frozen fine-tuning) and fixing
what made its student numbers unreliable.

| Problem in `iam_pretrain_finale_1` | Fix here |
|---|---|
| Student photos opened **without EXIF rotation**; the server rotates them (`app/utils/image_io.py`) | `open_image()` applies `exif_transpose` everywhere |
| Student fine-tune evaluated on **all** students, including the ones it trained on | Students split into train / val / test **by person**; test students are never trained on |
| Only top-1 reported; the app shows **5** suggestions | Top-1 / top-3 / top-5 / mean rank, ranked by **centroid** — the same rule as the API's `search_similar` |
| Contrastive loss on random pairs (mostly easy negatives) | **Batch-hard triplet loss** on cosine distance with P×K batches (Hermans et al., 2017) |
| Light augmentation; CVL scans look nothing like phone photos | **Phone-photo augmentation** (paper tone, ruled lines, lighting, shadow, blur, noise, JPEG, perspective) |
| Training on full CVL pages (~7 lines squashed to 224 px) vs 2–3-line phone captures | CVL pages are cut into random **2–3-line bands** during training |
| Checkpoint chosen by pair AUC | Chosen by **identification top-1** on validation writers |
| No comparison with the deployed model | Baseline vs new model on the **same test split**; export only if better |
| 310 CVL writers only | Optional **IAM** (~650 writers) |

**How to run:** set the paths in section 1, then *Run all* on a **GPU** runtime: Colab, or
Kaggle Notebooks (free GPU, about 30 h/week) when Colab's GPU limit is reached. Each epoch is saved
to `OUT_DIR/last.pth`; after a disconnect just *Run all* again and it carries on where it stopped.
To start over, delete `OUT_DIR/last.pth`. The last section tells you whether to deploy and how.

## 0. Setup

In [ ]:
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'tqdm'], check=True)

## 1. Configuration

In [ ]:
# CONFIG
import os
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_KAGGLE:
    # Upload CVL, the student folders and the backend's ml/best_model.pth as private Kaggle
    # Datasets, attach them to this notebook (Add Input), then fix the folder names below.
    INPUT = Path('/kaggle/input')
    CVL_ROOT = INPUT / 'cvl-database' / 'cvl-database-cropped-1-1'
    IAM_ROOT = INPUT / 'iam-handwriting'
    STUDENT_DATASET_PATH = INPUT / 'student-handwriting' / 'handwritting_dataset'
    BASELINE_PATH = INPUT / 'scriptflow-baseline' / 'best_model.pth'
    OUT_DIR = Path('/kaggle/working/writer_id_v2')
else:
    DRIVE = Path('/content/drive/MyDrive')
    CVL_ROOT = DRIVE / 'CVL/extracted/cvl-database-cropped-1-1'
    IAM_ROOT = DRIVE / 'IAM'            # needs ascii/forms.txt and lines/<a01>/<a01-000u>/*.png
    # One folder per student, named however you like (Agatha/, 2023004446/, ...).
    STUDENT_DATASET_PATH = DRIVE / 'handwritting_dataset/handwritting_dataset'
    # The checkpoint currently in the backend's ml/best_model.pth (upload it to Drive).
    BASELINE_PATH = DRIVE / 'models/cvl_improved.pth'
    OUT_DIR = DRIVE / 'models/writer_id_v2'

# --- Data -----------------------------------------------------------------
CVL_MIN_IMAGES = 5
COPY_CVL_TO_LOCAL = True                # Colab: Drive reads are slow; copy once per session
USE_IAM = False                         # True once IAM is uploaded
STUDENT_MIN_IMAGES = 3                  # leave-one-out needs >= 2; 3 gives a real gallery
STUDENT_SPLIT = (0.6, 0.2, 0.2)         # train / val / test, split by STUDENT
REQUIRE_STUDENTS = True                 # stop early if no student folders load

NEW_MODEL_VERSION = 'siamese-resnet18-v2'
RESUME = True                           # continue from OUT_DIR/last.pth after a disconnect
CPU_QUICK_PROFILE = True                # without a GPU, do a much smaller run (see setup output)

# --- Training ------------------------------------------------------------------
SEED = 42
P, K = 16, 4                            # writers x samples per batch (batch = 64)
STUDENT_SHARE = 0.25                    # share of each batch's writers taken from student-train
BATCHES_PER_EPOCH = 150
HEAD_EPOCHS = 8                         # stage 1: frozen backbone, train the head
FT_EPOCHS = 25                          # stage 2: unfreeze layer3 + layer4
PATIENCE = 6
MARGIN = 0.25                           # triplet margin on cosine distance (range 0..2)
UNFREEZE = (6, 7)                       # ResNet layer3, layer4
LR_HEAD, LR_BACKBONE = 1e-3, 1e-5
BAND_FRACTION = (0.25, 0.45)            # CVL page -> 2-3-line band
PHONE_AUG_P = 0.8                       # scans: usually make them look like phone photos
STUDENT_AUG_P = 0.3                     # student photos already are phone photos
NUM_WORKERS = min(4, os.cpu_count() or 2)

In [ ]:
import io
import json
import random
import shutil
import time
from collections import defaultdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter, ImageOps
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import models, transforms


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device, '| data workers:', NUM_WORKERS)

if device.type == 'cpu':
    print('WARNING: no GPU. The full run takes days on CPU and Colab will disconnect first.\n'
          '  Best: Kaggle Notebooks (free GPU, ~30 h/week) or wait for the Colab GPU limit to reset.')
    if CPU_QUICK_PROFILE:
        # Roughly 6x less work per epoch and far fewer epochs; only layer4 is fine-tuned.
        P, K, BATCHES_PER_EPOCH, HEAD_EPOCHS, FT_EPOCHS, UNFREEZE = 8, 4, 40, 3, 6, (7,)
        print(f'  CPU quick profile: P={P} K={K}, {BATCHES_PER_EPOCH} batches/epoch, '
              f'{HEAD_EPOCHS}+{FT_EPOCHS} epochs, unfreeze {UNFREEZE}. Weaker than a GPU run.')

## 2. Load the data

Every writer id is prefixed with its source (`cvl:`, `iam:`, `stu:`) so ids can never collide.
Each sample is an *item*: a CVL page, an IAM form (a list of its line images), or a student photo.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}


def open_image(path):
    """Open upright RGB. Phone cameras store rotation in EXIF instead of rotating the
    pixels. The server applies it on upload, so training and evaluation must too,
    otherwise the model is scored on sideways handwriting it will never see."""
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')


def load_cvl(root, min_images=CVL_MIN_IMAGES):
    writers = defaultdict(list)
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in {'.tif', '.tiff'}:
            writers[p.name.split('-')[0]].append(('page', p))   # 0201-4-cropped.tif -> 0201
    return {f'cvl:{w}': v for w, v in writers.items() if len(v) >= min_images}


def load_iam(root):
    root = Path(root)
    form_writer = {}
    for line in (root / 'ascii' / 'forms.txt').read_text().splitlines():
        if line.strip() and not line.startswith('#'):
            form_id, writer_id = line.split()[:2]
            form_writer[form_id] = writer_id
    writers = defaultdict(list)
    for form_dir in sorted((root / 'lines').glob('*/*')):
        lines = sorted(form_dir.glob('*.png'))
        writer = form_writer.get(form_dir.name)
        if writer and len(lines) >= 2:
            writers[f'iam:{writer}'].append(('lines', lines))
    return dict(writers)


def load_students(root, min_images=STUDENT_MIN_IMAGES):
    root = Path(root)
    if not root.exists():
        print('Student folder not found:', root)
        return {}
    writers, skipped = {}, {}
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        imgs = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)
        if len(imgs) >= min_images:
            writers[f'stu:{folder.name}'] = [('photo', p) for p in imgs]
        else:
            skipped[folder.name] = len(imgs)
    if skipped:
        print(f'Skipped {len(skipped)} student(s) with < {min_images} images:', skipped)
    return writers


if IN_COLAB and COPY_CVL_TO_LOCAL:
    local = Path('/content/CVL_local')
    if not local.exists():
        shutil.copytree(CVL_ROOT, local)
    CVL_ROOT = local

cvl = load_cvl(CVL_ROOT)
iam = load_iam(IAM_ROOT) if USE_IAM else {}
students = load_students(STUDENT_DATASET_PATH)
all_writers = {**cvl, **iam, **students}

for name, d in [('CVL', cvl), ('IAM', iam), ('Students', students)]:
    counts = [len(v) for v in d.values()]
    summary = f'min {min(counts)}, median {int(np.median(counts))}, max {max(counts)}' if counts else '-'
    print(f'{name:9s} writers: {len(d):4d}   items: {sum(counts):5d}   per writer: {summary}')

if not students and REQUIRE_STUDENTS:
    root = Path(STUDENT_DATASET_PATH)
    found = sorted(p.name for p in root.iterdir())[:10] if root.exists() else 'folder does not exist'
    raise RuntimeError(
        f'No usable student folders under {root} (first entries: {found}).\n'
        'Expected one sub-folder per student with >= STUDENT_MIN_IMAGES images each. Without students\n'
        'the model cannot be tested on real phone photos. Fix STUDENT_DATASET_PATH, or set\n'
        'REQUIRE_STUDENTS = False to train on public data only.')

### Student data audit

How many student photos carry an EXIF rotation that `iam_pretrain_finale_1` ignored, and which
students have too few samples. The backend needs 10 per student before it counts them as enrolled.

In [ ]:
def exif_orientation(path):
    try:
        with Image.open(path) as im:
            return im.getexif().get(0x0112, 1)
    except Exception:
        return None


photos = [p for items in students.values() for _, p in items]
rotated = [p for p in photos if exif_orientation(p) not in (None, 1)]
print(f'Student photos: {len(photos)}   with EXIF rotation: {len(rotated)} '
      f'({len(rotated) / max(len(photos), 1):.0%}) -- these were sideways in the old notebook')
counts = sorted(((len(v), w) for w, v in students.items()))
print('Fewest samples:', counts[:5])
print('Students with >= 10 samples (backend "enrolled"):', sum(n >= 10 for n, _ in counts), '/', len(counts))

## 3. Preprocessing — must stay identical to `ml/transforms.py`

`prepare_handwriting`, `tensorize_crop` and `get_eval_crops` are copied from the backend. If you change
them here, change them there too, or the server will embed images differently from how the model was
trained and evaluated.

In [ ]:
TARGET_H = 224
CROP_SIZE = 224


def prepare_handwriting(image, target_height=TARGET_H, min_width=CROP_SIZE):
    """Crop ink, scale HEIGHT to 224, keep aspect ratio."""
    image = image.convert('RGB')
    gray = image.convert('L')
    bbox = ImageOps.invert(gray).getbbox()
    if bbox:
        image = image.crop(bbox)
    width, height = image.size
    height = max(height, 1)
    new_w = max(min_width, int(round(width * target_height / height)))
    image = image.resize((new_w, target_height), Image.BILINEAR)
    if new_w < min_width:
        pad = min_width - new_w
        left = pad // 2
        image = ImageOps.expand(image, border=(left, 0, pad - left, 0), fill=(255, 255, 255))
    return image


def tensorize_crop(pil_img):
    """Turn one already-224x224 PIL crop into a normalized tensor."""
    img = pil_img.convert('RGB')
    img = transforms.Grayscale(num_output_channels=3)(img)
    tensor = transforms.ToTensor()(img)
    return transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(tensor)


def get_eval_crops(image, n_crops=4):
    """4 evenly spaced horizontal 224x224 crops -- what the server embeds."""
    prepared = prepare_handwriting(image)
    width, height = prepared.size
    if width <= CROP_SIZE:
        return [prepared.crop((0, 0, CROP_SIZE, CROP_SIZE))]
    xs = np.linspace(0, width - CROP_SIZE, num=n_crops)
    return [prepared.crop((int(x), 0, int(x) + CROP_SIZE, CROP_SIZE)) for x in xs]

## 4. Make training samples look like what the app sends

* **CVL pages** become random horizontal bands of roughly 2–3 lines (70% of the time).
* **IAM forms** become 2–3 consecutive lines stacked into one block.
* **Scans** are then made to look like phone photos. Student photos get much lighter treatment.

In [ ]:
def stack_lines(paths, gap=12):
    ims = [open_image(p) for p in paths]
    canvas = Image.new('RGB', (max(i.width for i in ims), sum(i.height for i in ims) + gap * (len(ims) - 1)), 'white')
    y = 0
    for im in ims:
        canvas.paste(im, (0, y))
        y += im.height + gap
    return canvas


def random_band(page, frac=BAND_FRACTION):
    bbox = ImageOps.invert(page.convert('L')).getbbox()
    if bbox:
        page = page.crop(bbox)
    w, h = page.size
    band_h = max(1, int(h * random.uniform(*frac)))
    y = random.randint(0, max(0, h - band_h))
    return page.crop((0, y, w, y + band_h))


def load_block(item, train):
    kind, payload = item
    if kind == 'lines':
        n = min(len(payload), random.choice([2, 3])) if train else min(len(payload), 3)
        start = random.randint(0, len(payload) - n) if train else 0
        return stack_lines(payload[start:start + n])
    image = open_image(payload)
    if kind == 'page' and train and random.random() < 0.7:
        return random_band(image)
    return image


class PhoneAugment:
    """Paper tone, ruled lines, uneven lighting, shadow, noise, resolution loss, blur, JPEG.
    Applied after prepare_handwriting so stroke scale matches what the server sees."""

    def __init__(self, p):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        arr = np.asarray(img.convert('RGB'), dtype=np.float32) / 255.0
        if random.random() < 0.6:                                  # paper tone
            arr *= np.array([random.uniform(0.85, 1.0), random.uniform(0.82, 1.0), random.uniform(0.75, 0.98)])
        if random.random() < 0.4:                                  # ruled lines
            spacing = random.randint(28, 48)
            tint = np.array([random.uniform(0.55, 0.8), random.uniform(0.65, 0.85), random.uniform(0.8, 0.95)])
            for y in range(random.randint(0, spacing), h, spacing):
                arr[y:y + random.randint(1, 2)] *= tint
        if random.random() < 0.6:                                  # lighting gradient
            if random.random() < 0.5:
                arr *= np.linspace(random.uniform(0.6, 1.0), random.uniform(0.85, 1.1), w)[None, :, None]
            else:
                arr *= np.linspace(random.uniform(0.6, 1.0), random.uniform(0.85, 1.1), h)[:, None, None]
        if random.random() < 0.3:                                  # shadow over one side
            x0 = random.randint(0, w - 1)
            if random.random() < 0.5:
                arr[:, :x0] *= random.uniform(0.5, 0.8)
            else:
                arr[:, x0:] *= random.uniform(0.5, 0.8)
        if random.random() < 0.5:                                  # sensor noise
            arr += np.random.normal(0, random.uniform(0.01, 0.04), arr.shape)
        img = Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))
        if random.random() < 0.4:                                  # resolution loss
            s = random.uniform(0.4, 0.8)
            img = img.resize((max(1, int(w * s)), max(1, int(h * s))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        if random.random() < 0.4:                                  # focus blur
            img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.2)))
        if random.random() < 0.7:                                  # JPEG
            buf = io.BytesIO()
            img.save(buf, 'JPEG', quality=random.randint(40, 90))
            buf.seek(0)
            img = Image.open(buf).convert('RGB')
        return img


def make_train_transform(aug_p):
    return transforms.Compose([
        transforms.Lambda(prepare_handwriting),
        PhoneAugment(aug_p),
        transforms.RandomPerspective(distortion_scale=0.15, p=0.3, fill=255),
        transforms.RandomAffine(degrees=3, translate=(0.02, 0.02), shear=3, fill=255),
        transforms.RandomCrop(CROP_SIZE),
        transforms.Lambda(tensorize_crop),
    ])


scan_transform = make_train_transform(PHONE_AUG_P)
student_transform = make_train_transform(STUDENT_AUG_P)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]


def show_views(item, transform, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax in axes:
        t = transform(load_block(item, train=True))
        ax.imshow((t * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy())
        ax.axis('off')
    fig.suptitle(title)
    plt.show()


# Sanity check: writing must stay readable after augmentation.
if cvl:
    show_views(next(iter(cvl.values()))[0], scan_transform, 'CVL band -> phone-style training views')
if students:
    show_views(next(iter(students.values()))[0], student_transform, 'Student photo training views')

## 5. Split writers

Splits are by **writer**, never by image. Each writer's split comes from a hash of its id, so it never
changes between runs, and students added later are spread over train / val / test instead of all landing
in training. IAM is used only for training. The split is saved to `OUT_DIR/splits.json` for the record.

In [ ]:
import hashlib


def split_writers(ids, fracs, seed=SEED):
    parts = ([], [], [])
    for w in sorted(ids):
        u = int(hashlib.md5(f'{seed}:{w}'.encode()).hexdigest(), 16) / 16 ** 32   # stable, in [0, 1)
        parts[0 if u < fracs[0] else 1 if u < fracs[0] + fracs[1] else 2].append(w)
    return parts


cvl_tr, cvl_va, cvl_te = split_writers(cvl, (0.8, 0.1, 0.1))
stu_tr, stu_va, stu_te = split_writers(students, STUDENT_SPLIT)
splits = {'cvl_train': cvl_tr, 'cvl_val': cvl_va, 'cvl_test': cvl_te,
          'stu_train': stu_tr, 'stu_val': stu_va, 'stu_test': stu_te}
(OUT_DIR / 'splits.json').write_text(json.dumps(splits, indent=1))

train_ids = splits['cvl_train'] + list(iam) + splits['stu_train']
for k, v in splits.items():
    print(f'{k:10s} {len(v):4d} writers')
print(f'Training writers in total: {len(train_ids)}')
if len(splits['stu_test']) < 5:
    print('WARNING: fewer than 5 test students -- student numbers will be very noisy. Collect more data.')

## 6. P×K batches, model and batch-hard triplet loss

Each batch holds **P writers × K samples**, a fixed share of them students, so every anchor has
positives and hard negatives. The loss uses **cosine distance**, the same distance the server ranks by.
The network is the backend's `EmbeddingNetwork`; parameter names are identical, so the exported
`state_dict` loads straight into `ml/model.py`.

In [ ]:
class WriterItems(Dataset):
    def __init__(self, writers, ids):
        self.items, self.by_writer = [], defaultdict(list)
        self.label_of = {w: i for i, w in enumerate(ids)}
        for w in ids:
            for item in writers[w]:
                self.by_writer[w].append(len(self.items))
                self.items.append((w, item))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        writer, item = self.items[index]
        transform = student_transform if writer.startswith('stu:') else scan_transform
        return transform(load_block(item, train=True)), self.label_of[writer]


class PKSampler(Sampler):
    def __init__(self, by_writer, p, k, batches, student_share):
        self.by_writer, self.p, self.k, self.batches = by_writer, p, k, batches
        self.students = [w for w in by_writer if w.startswith('stu:')]
        self.others = [w for w in by_writer if not w.startswith('stu:')]
        self.n_students = min(len(self.students), round(p * student_share)) if self.others else min(len(self.students), p)

    def __iter__(self):
        for _ in range(self.batches):
            chosen = random.sample(self.students, self.n_students)
            chosen += random.sample(self.others, min(len(self.others), self.p - len(chosen)))
            batch = []
            for w in chosen:
                idx = self.by_writer[w]
                batch += random.sample(idx, self.k) if len(idx) >= self.k else random.choices(idx, k=self.k)
            yield batch

    def __len__(self):
        return self.batches


train_set = WriterItems(all_writers, train_ids)
train_loader = DataLoader(
    train_set,
    batch_sampler=PKSampler(train_set.by_writer, P, K, BATCHES_PER_EPOCH, STUDENT_SHARE),
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
    persistent_workers=NUM_WORKERS > 0,
)


class EmbeddingNetwork(nn.Module):
    def __init__(self, embedding_dim=256, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = models.resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.embedding_head = nn.Sequential(nn.Linear(512, embedding_dim), nn.BatchNorm1d(embedding_dim))
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_layers(self, indices):
        for i in indices:
            for p in self.backbone[i].parameters():
                p.requires_grad = True
        for m in self.backbone.modules():          # keep ImageNet BatchNorm statistics
            if isinstance(m, nn.BatchNorm2d):
                for p in m.parameters():
                    p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        for m in self.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def forward(self, x):
        x = self.backbone(x).flatten(1)
        return F.normalize(self.embedding_head(x), p=2, dim=1)


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_dim=256, pretrained=True):
        super().__init__()
        self.embedding_network = EmbeddingNetwork(embedding_dim, pretrained)

    def forward(self, img1, img2):
        return self.embedding_network(img1), self.embedding_network(img2)

    def get_embedding(self, img):
        return self.embedding_network(img)


def batch_hard_triplet(emb, labels, margin=MARGIN):
    dist = 1.0 - emb @ emb.t()                                    # cosine distance
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)
    hardest_pos = dist.masked_fill(~same | eye, -1.0).max(1).values
    hardest_neg = dist.masked_fill(same, 3.0).min(1).values
    loss = F.relu(hardest_pos - hardest_neg + margin).mean()
    return loss, (hardest_pos < hardest_neg).float().mean().item()

## 7. Evaluation — the same way the server identifies

Embeddings are made exactly like `get_embedding_from_image` (4 crops, averaged, normalised).
Each image is used as a query against every writer's **centroid** with that image left out, exactly
like `search_similar` in the API. `vs all students` adds every other student as a distractor,
which is closer to a real class roster.

In [ ]:
@torch.no_grad()
def embed_writers(model, ids):
    model.eval()
    vecs, labels = [], []
    for w in ids:
        for item in all_writers[w]:
            crops = get_eval_crops(load_block(item, train=False))
            batch = torch.stack([tensorize_crop(c) for c in crops]).to(device)
            emb = model.get_embedding(batch).float()
            vecs.append(F.normalize(emb.mean(0, keepdim=True), dim=1).cpu())
            labels.append(w)
    return torch.cat(vecs), np.array(labels)


def identification_metrics(embs, labels, query_writers=None):
    """Leave-one-out identification ranked by writer centroid. Queries are images of
    query_writers (default: all); every writer present is in the gallery."""
    writers = np.unique(labels)
    members = {w: np.where(labels == w)[0] for w in writers}
    sums = torch.stack([embs[members[w]].sum(0) for w in writers])
    query_writers = set(writers if query_writers is None else query_writers)
    ranks = []
    for i, own in enumerate(labels):
        if own not in query_writers or len(members[own]) < 2:
            continue
        protos = sums.clone()
        j = int(np.where(writers == own)[0][0])
        protos[j] -= embs[i]                                     # leave the query out
        scores = F.normalize(protos, dim=1) @ embs[i]
        order = writers[torch.argsort(scores, descending=True).numpy()]
        ranks.append(int(np.where(order == own)[0][0]) + 1)
    ranks = np.array(ranks)
    if len(ranks) == 0:
        return {}
    q = np.where(np.isin(labels, list(query_writers)))[0]
    sim = (embs[q] @ embs[q].t()).numpy()
    iu = np.triu_indices(len(q), 1)
    same = (labels[q][:, None] == labels[q][None, :])[iu]
    return {
        'top1': float((ranks <= 1).mean()), 'top3': float((ranks <= 3).mean()),
        'top5': float((ranks <= 5).mean()), 'mean_rank': float(ranks.mean()),
        'pair_auc': float(roc_auc_score(same, sim[iu])) if same.any() and (~same).any() else float('nan'),
        'queries': int(len(ranks)), 'gallery': int(len(writers)),
    }


def evaluate(model, sets):
    out = {}
    for name, (ids, query) in sets.items():
        if not ids:
            continue
        embs, labels = embed_writers(model, ids)
        m = identification_metrics(embs, labels, query)
        if m:
            out[name] = m
    return out


VAL_SETS = {
    'cvl_val': (splits['cvl_val'], None),
    'stu_val': (splits['stu_val'], None),
}
TEST_SETS = {
    'cvl_test': (splits['cvl_test'], None),
    'stu_test': (splits['stu_test'], None),
    'stu_test vs all students': (splits['stu_train'] + splits['stu_val'] + splits['stu_test'], splits['stu_test']),
}


def fmt(metrics):
    return ' | '.join(f"{k} top1 {m['top1']:.1%} top5 {m['top5']:.1%}" for k, m in metrics.items())


def selection_score(metrics):
    """Mean top-1 over the validation sets; students count when there are any."""
    tops = [m['top1'] for m in metrics.values()]
    return float(np.mean(tops)) if tops else 0.0

## 8. Train

In [ ]:
model = SiameseNetwork().to(device)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
best_path = OUT_DIR / 'best_model.pth'
last_path = OUT_DIR / 'last.pth'        # resume point, rewritten after every epoch
STAGES = ['head', 'finetune']

resume = torch.load(last_path, map_location=device, weights_only=False) if RESUME and last_path.exists() else None
if resume:
    model.load_state_dict(resume['model'])
    best_score, history = resume['best_score'], resume['history']
    print(f"Resuming from {last_path}: stage {resume['stage']} epoch {resume['epoch']} "
          f"({'finished' if resume['done'] else 'in progress'}), best validation top-1 {best_score:.1%}")
else:
    best_score, history = -1.0, []
    print('ImageNet only (before training):', fmt(evaluate(model, VAL_SETS)))


def save_last(stage, epoch, done, bad, optimizer=None, scheduler=None):
    torch.save({
        'stage': stage, 'epoch': epoch, 'done': done, 'bad': bad,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict() if optimizer else None,
        'scheduler': scheduler.state_dict() if scheduler else None,
        'scaler': scaler.state_dict(),
        'best_score': best_score, 'history': history,
    }, last_path)


def run_stage(stage, epochs):
    global best_score
    first_epoch, bad, state = 1, 0, None
    if resume:
        at, here = STAGES.index(resume['stage']), STAGES.index(stage)
        if at > here or (at == here and resume['done']):
            print(f'{stage}: already finished, skipping')
            return
        if at == here:
            first_epoch, bad, state = resume['epoch'] + 1, resume['bad'], resume

    net = model.embedding_network
    if stage == 'head':
        optimizer = torch.optim.AdamW(net.embedding_head.parameters(), lr=LR_HEAD, weight_decay=1e-4)
    else:
        net.unfreeze_layers(UNFREEZE)
        optimizer = torch.optim.AdamW([
            {'params': [p for p in net.backbone.parameters() if p.requires_grad], 'lr': LR_BACKBONE},
            {'params': net.embedding_head.parameters(), 'lr': LR_HEAD / 10},
        ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    if state:
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
    params = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(first_epoch, epochs + 1):
        if bad >= PATIENCE:
            break
        model.train()
        t0, losses, fracs = time.time(), [], []
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                emb = model.get_embedding(x)
            loss, frac = batch_hard_triplet(emb.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(loss.item())
            fracs.append(frac)
        val = evaluate(model, VAL_SETS)
        score = selection_score(val)
        history.append({'stage': stage, 'epoch': epoch, 'loss': float(np.mean(losses)), 'score': score, **{
            f'{k}_{m}': v[m] for k, v in val.items() for m in ('top1', 'top5')}})
        print(f'{stage} {epoch:02d} | loss {np.mean(losses):.4f} | hard-triplet ok {np.mean(fracs):.0%} '
              f'| {fmt(val)} | {time.time() - t0:.0f}s')
        if score > best_score:
            best_score, bad = score, 0
            torch.save(model.state_dict(), best_path)
            print('  saved (best validation top-1)')
        else:
            bad += 1
            if bad >= PATIENCE:
                print('  early stopping')
        save_last(stage, epoch, False, bad, optimizer, scheduler)

    model.load_state_dict(torch.load(best_path, map_location=device))
    save_last(stage, epochs, True, 0)


run_stage('head', HEAD_EPOCHS)
run_stage('finetune', FT_EPOCHS)
print(f'Best validation score: {best_score:.1%}')

In [ ]:
if history:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot([h['loss'] for h in history])
    ax[0].set_title('Training loss')
    for key in [k for k in history[0] if k.endswith('_top1')]:
        ax[1].plot([h.get(key, np.nan) for h in history], label=key)
    ax[1].set_title('Validation top-1')
    ax[1].legend()
    for a in ax:
        a.set_xlabel('epoch (both stages)')
    plt.show()

## 9. New model vs the deployed baseline — same test writers

`stu_test` is the number that matters: real phone photos of students the model never trained on.

In [ ]:
new_model = SiameseNetwork(pretrained=False).to(device)
new_model.load_state_dict(torch.load(best_path, map_location=device))
results = {'new': evaluate(new_model, TEST_SETS)}

if Path(BASELINE_PATH).exists():
    base_model = SiameseNetwork(pretrained=False).to(device)
    base_model.load_state_dict(torch.load(BASELINE_PATH, map_location=device))
    results['baseline'] = evaluate(base_model, TEST_SETS)
else:
    print('Baseline not found at', BASELINE_PATH, '-- upload the backend ml/best_model.pth to compare.')

# A Markdown table you can paste into the report (Chapter 4).
rows = ['| Test set | Model | Writers | Queries | Top-1 | Top-3 | Top-5 | Mean rank | Pair AUC |',
        '|---|---|---|---|---|---|---|---|---|']
for name in TEST_SETS:
    for who in ('baseline', 'new'):
        m = results.get(who, {}).get(name)
        if m:
            rows.append(f"| {name} | {who} | {m['gallery']} | {m['queries']} | {m['top1']:.1%} | {m['top3']:.1%} "
                        f"| {m['top5']:.1%} | {m['mean_rank']:.2f} | {m['pair_auc']:.3f} |")
results_table = '\n'.join(rows)
print(results_table)

## 10. Export

Recommends deploying only if the new model is at least as good as the baseline on
`stu_test vs all students` (top-1, then top-5 as a tie-break) -- unseen students ranked against the
whole student pool, the closest thing to a real class. To deploy:

1. Copy `OUT_DIR/best_model.pth` over the backend's `ml/best_model.pth`.
2. Set `MODEL_VERSION` in the backend `.env` to the value printed below.
3. Restart the API, then run `python -m scripts.reembed` in the backend folder.

In [ ]:
def key(m):
    return (m['top1'], m['top5']) if m else (-1, -1)


# Judge on the realistic set: unseen students ranked against every student.
DECISION_SET = 'stu_test vs all students'
new_s = results['new'].get(DECISION_SET)
base_s = results.get('baseline', {}).get(DECISION_SET)
if new_s is None:
    print('No student test set -- cannot judge on real phone photos. Collect student data first.')
    deploy = False
else:
    deploy = base_s is None or key(new_s) >= key(base_s)

meta = {
    'model_version': NEW_MODEL_VERSION,
    'trained_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'deploy_recommended': deploy,
    'results': results,
    'config': {k: (str(v) if isinstance(v, Path) else v) for k, v in {
        'P': P, 'K': K, 'MARGIN': MARGIN, 'UNFREEZE': UNFREEZE, 'LR_HEAD': LR_HEAD,
        'LR_BACKBONE': LR_BACKBONE, 'BAND_FRACTION': BAND_FRACTION, 'PHONE_AUG_P': PHONE_AUG_P,
        'USE_IAM': USE_IAM, 'STUDENT_SPLIT': STUDENT_SPLIT, 'SEED': SEED}.items()},
    'splits': splits,
    'history': history,
}
(OUT_DIR / 'meta.json').write_text(json.dumps(meta, indent=1))
(OUT_DIR / 'results.md').write_text(results_table + '\n')

if deploy:
    print(f'DEPLOY: {best_path}\n  then set MODEL_VERSION={NEW_MODEL_VERSION} and run: python -m scripts.reembed')
else:
    print('Do NOT deploy: the baseline is as good or better on unseen students. Keep ml/best_model.pth.')
print('Saved', OUT_DIR / 'meta.json', 'and', OUT_DIR / 'results.md')